# Web-Scraping the BRAVA Data

Amber has an example using OGLE data in her mini-course notebook: https://github.com/rges-pit/minicourses/blob/main/chapter5/Chapter5.ipynb

In [1]:
from io import StringIO
import bs4 as bs
import urllib
import urllib.request
import pandas as pd
import numpy as np
import sys
import ssl

from urllib.error import HTTPError
from time import sleep

In [2]:
datadir = "/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing"
catdir = f"{datadir}/catalogs/BRAVA_test"
tabdir = f"{datadir}/tables"
faileddir = f"{datadir}/failed_tables"

## Load Main Table

Copy-pasted this from the page, but could theoretically scrape it too. Can use it to get the l and b coordinates that need to be fed into the URL generation, since they all seem to be named as `https://brava.astro.ucla.edu/DATA/{int(l)}{int(b)}.html.`

In [3]:
tab = pd.read_csv(f"{tabdir}/BRAVA_Main_Table.no_dup.txt", sep=r'\s+', header=0)
print(tab.shape)
print(tab.columns)

(80, 8)
Index(['l_deg', 'b_deg', 'mean_Vgc', 'err_mean_Vgc', 'sigma', 'err_sigma',
       'Na', 'Nb'],
      dtype='object')


In [4]:
display(tab)

,l_deg,b_deg,mean_Vgc,err_mean_Vgc,sigma,err_sigma,Na,Nb
0,0,4.0,7.7,11.6,112.9,8.2,95,95.0
1,0,-1.0,5.8,15.0,135.4,10.6,82,97.0
2,0,-2.0,-8.1,12.1,125.8,8.6,108,NaN
3,-6,-3.0,-58.0,11.5,91.0,8.1,63,NaN
4,0,-3.0,-20.4,11.9,121.6,8.4,105,107.0
...,...,...,...,...,...,...,...,...
75,6,-8.0,51.0,7.4,72.7,5.2,98,NaN
76,7,-8.0,47.4,7.6,76.2,5.3,102,NaN
77,8,-8.0,58.1,6.9,68.5,4.9,98,NaN
78,9,-8.0,54.9,7.0,72.2,4.9,108,NaN


## Define Web-Scraping + Data Manipulation Functions

In [5]:
# To define the unique link referenced on the main DATA page
def get_data_url(l, b):
    if l < 0:
        lstr = f"m{int(np.abs(l))}"
    elif l == 0:
        lstr = f"{int(l)}"
    else:
        lstr = f"p{int(l)}"

    if b < 0:
        bstr = f"m{int(np.abs(b))}"
    else:
        bstr = f"p{int(b)}"

    url = f"https://brava.astro.ucla.edu/DATA/{lstr}{bstr}.html"
    return url

In [6]:
# To fetch the data
# Could make this account for URLErrors in a similar way, but not needed in this case
def fetch_table_data(url, tab_idx=5, max_retries=3):
    # Kept getting an SSL Certificate Error. Giving it this "context" solves that.
    context = ssl._create_unverified_context()
    
    # Add exceptions for HTTP Error
    retries = 0
    while retries <= max_retries:
        try:
            # try to do everything normally
            source = urllib.request.urlopen(url, context=context).read()
            soup = bs.BeautifulSoup(source, 'lxml')
            table = soup.find_all('table')
            df = pd.read_html(StringIO(str(table)))[tab_idx]
            return df
        
        except HTTPError as e:
            # Account for the main HTTP Errors, give up if it isn't either of those
            if e.code == 404:
                print(f"HTTP Error 404 occurred. Table is probably named inconsistently bc why make life easy.")
                print(f"Failed URL {url}")
                break
            elif e.code == 502:
                retries += 1
                print(f"Retry {retries}/{max_retries} after 502 Error...")
                sleep(2)
            else:
                print(f"An unusual HTTP Error occurred: {e}")
                print(f"Failed URL: {url}")
                break
    

In [9]:
# To reformat the data
# There are probably more elegant ways to do this, but I couldn't be bothered
def reformat_brava(table):
    old_cols = ['Aperture', 'lb', 'RADec', 'J', 'H', 'K', 'Vhc_kmps', 'E(B-V)', 
                'J0', 'H0', 'K0', 'TiO', 'Spectrum', '2MASS_ID', 
                'l_deg', 'b_deg', 'RA_deg', 'Dec_deg']
    new_cols = ['Aperture', 'l_deg', 'b_deg', 'RA_deg', 'Dec_deg', 'J', 'H', 'K', 
                'Vhc_kmps', 'E(B-V)', 'J0', 'H0', 'K0', 'TiO', '2MASS_ID']

    table.drop(0, axis=0, inplace=True) # to drop first row  which contains original header
    new_table = pd.concat([table, table.iloc[:,1].str.split(',', expand=True)], axis=1) # split l and b
    new_table = pd.concat([new_table, table.iloc[:,2].str.split(',', expand=True)], axis=1) # split RA and Dec
    new_table.columns = old_cols # to give a header to new df with expanded coords
    final_table = new_table[new_cols] # to rearrange + drop unnecessary
    final_table.reset_index(drop=True, inplace=True)
    # make sure l,b,r,d are floats to avoid messing up anything
    final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)
    return final_table    

In [11]:
# Make lists to save the coordinates + URLs in case it's a lot of them
failed_ls = []
failed_bs = []
failed_urls = []

# Loop over main data table
for i, row in tab.iterrows():
    l = row['l_deg']
    b = row['b_deg']

    # Create the URL from the l and b coordinates
    url = get_data_url(l,b)
    print(url) # I only leave this print statement to show how much progress it's making

    # Scrape for table
    table = fetch_table_data(url)

    # Reformat + save if it was successful
    if table is not None:
        table = reformat_brava(table)
        print(f"{tabdir}/BRAVA_l{l:0.3f}_b{b:0.3f}.table")
        table.to_csv(f"{tabdir}/BRAVA_l{l:0.3f}_b{b:0.3f}.table", sep=' ', header=True, index=False, na_rep='NaN')

    # Add the failed coords/urls to the above lists if unsuccessful
    else:
        failed_ls.append(l)
        failed_bs.append(b)
        failed_urls.append(url)

https://brava.astro.ucla.edu/DATA/0p4.html


/tmp/ipykernel_1558860/1548505758.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)


/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing/tables/BRAVA_l0.000_b4.000.table
https://brava.astro.ucla.edu/DATA/0m1.html


/tmp/ipykernel_1558860/1548505758.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)


/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing/tables/BRAVA_l0.000_b-1.000.table
https://brava.astro.ucla.edu/DATA/0m2.html


...

/tmp/ipykernel_1558860/1548505758.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)


/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing/tables/BRAVA_l8.000_b-8.000.table
https://brava.astro.ucla.edu/DATA/p9m8.html


/tmp/ipykernel_1558860/1548505758.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)


/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing/tables/BRAVA_l9.000_b-8.000.table
https://brava.astro.ucla.edu/DATA/p10m8.html
/home/ast-gaudi-group/crisp/synthpop/BRAVA_kinematics_testing/tables/BRAVA_l10.000_b-8.000.table


/tmp/ipykernel_1558860/1548505758.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']] = final_table[['l_deg', 'b_deg', 'RA_deg', 'Dec_deg']].apply(pd.to_numeric)


In [12]:
print(failed_urls)

['https://brava.astro.ucla.edu/DATA/0m4.html']


## Reformat Failed

I have manually obtained the failed ones, now will correct as with the others. I could use the ls and bs saved above to make this into an automatic loop situation, but since it's only two of them, I don't really see the point.  

~~For the record, I think these two failed because they seem to have split their (0, -4) catalog into two separate catalogs. One is named m0m4.html, and the other is named 0m3_5.html. Maybe this was justified in the paper, but I don't remember the reasoning off the top of my head, and it isn't really relevant.~~

Corrected one of the tables listed as 0,-4 by changing it to 0, -3.5 in the main brava table, as it's named p0m3.5 on the website.

In [13]:
old_cols = ['Aperture', 'lb', 'RADec', 'J', 'H', 'K', 'Vhc_kmps', 'E(B-V)', 
            'J0', 'H0', 'K0', 'TiO', 'Spectrum', '2MASS_ID', 
            'l_deg', 'b_deg', 'RA_deg', 'Dec_deg']
new_cols = ['Aperture', 'l_deg', 'b_deg', 'RA_deg', 'Dec_deg', 'J', 'H', 'K', 
            'Vhc_kmps', 'E(B-V)', 'J0', 'H0', 'K0', 'TiO', '2MASS_ID']

### l,b = 0,-4

In [14]:
failed1 = pd.read_csv(f"{faileddir}/BRAVA_l0_b-4.table.failed", sep='\t', header=0)

# Don't need to drop row since headers are as they should be
# But will just do the rest as above since I know it works
failed1 = pd.concat([failed1, failed1.iloc[:,1].str.split(',', expand=True)], axis=1) # split l and b
failed1 = pd.concat([failed1, failed1.iloc[:,2].str.split(',', expand=True)], axis=1) # split RA and Dec
failed1.columns = old_cols # to give a header to new df with expanded coords
corrected1 = failed1[new_cols] # to rearrange + drop unnecessary
corrected1.reset_index(drop=True, inplace=True)

print(corrected1)
corrected1.to_csv(f"{tabdir}/BRAVA_l0.000_b-4.000.table", sep=' ', header=True, index=False, na_rep='NaN')

     Aperture    l_deg      b_deg    RA_deg     Dec_deg       J      H      K  \
0           1   0.0058   -3.7546   270.1409   -30.8358   10.141  9.036  8.693   
1           2   0.0680   -4.1923   270.6208   -30.9968   10.937  9.834  9.405   
2           3  -0.1879   -4.0272   270.3067   -31.1382   10.493  9.369  8.901   
3           4  -0.0113   -3.8847   270.2630   -30.9148   10.627  9.500  9.067   
4           7   0.1169   -3.8291   270.2800   -30.7761   10.037  8.874  8.487   
..        ...      ...        ...       ...         ...     ...    ...    ...   
105       137  -0.0427   -3.8945   270.2550   -30.9468   10.961  9.855  9.461   
106       138   0.0471   -4.2007   270.6175   -31.0191   10.697  9.631  9.134   
107       139  -0.0860   -4.0079   270.3454   -31.0403   10.922  9.874  9.436   
108       140  -0.0256   -3.7789   270.1476   -30.8751   10.368  9.335  9.058   
109       141   0.1212   -3.8380   270.2914   -30.7768   10.231  9.189  8.853   

     Vhc_kmps  E(B-V)      

### l,b = 0,-2

And I found out while doing other tests that the (0, -2) one is some sort of funky, so checking that one. Not sure what's going wrong exactly. But I'll go through it step-by-step to see where things are going wrong exactly. Scraping the table didn't fail, but it seems that whatever was scraped was wrong in some way. Copy-pasting into a file and then fixing it as above worked fine.

In [27]:
failed2 = pd.read_csv(f"{faileddir}/BRAVA_l0_b-4.table.failed", sep='\t', header=0)
display(failed2)

# Split coordinates, needs to happen with load so there aren't a bunch of extra columns if I need to re-run cells
failed2 = pd.concat([failed2, failed2.iloc[:,1].str.split(',', expand=True)], axis=1) # split l and b
failed2 = pd.concat([failed2, failed2.iloc[:,2].str.split(',', expand=True)], axis=1) # split RA and Dec

display(failed2)

,Aperture,"(l,b)(deg)","RA,Dec (J2000.0)",J (mag),H (mag),K (mag),Vhc (km/s),E(B-V),J0(mag),H0(mag),K0(mag),TiO(mag),Spectrum (fits),2MASS ID
0,1,"0.0058, -3.7546","270.1409, -30.8358",10.141,9.036,8.693,-67.0,1.108,9.141,8.398,8.286,-0.0316,Spectrum,18003382-3050089
1,2,"0.0680, -4.1923","270.6208, -30.9968",10.937,9.834,9.405,10.2,0.844,10.176,9.348,9.095,0.0971,Spectrum,18022900-3059483
2,3,"-0.1879, -4.0272","270.3067, -31.1382",10.493,9.369,8.901,108.3,0.867,9.711,8.870,8.583,0.4710,Spectrum,18011360-3108176
3,4,"-0.0113, -3.8847","270.2630, -30.9148",10.627,9.500,9.067,43.3,0.979,9.744,8.936,8.708,0.4769,Spectrum,18010312-3054534
4,7,"0.1169, -3.8291","270.2800, -30.7761",10.037,8.874,8.487,110.5,0.974,9.158,8.313,8.129,0.4879,Spectrum,18010719-3046338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,137,"-0.0427, -3.8945","270.2550, -30.9468",10.961,9.855,9.461,49.6,0.969,10.087,9.297,9.105,0.1758,Spectrum,18010118-3056486
106,138,"0.0471, -4.2007","270.6175, -31.0191",10.697,9.631,9.134,1.4,0.829,9.949,9.154,8.830,0.4380,Spectrum,18022820-3101087
107,139,"-0.0860, -4.0079","270.3454, -31.0403",10.922,9.874,9.436,202.4,0.869,10.138,9.373,9.117,0.0968,Spectrum,18012290-3102249
108,140,"-0.0256, -3.7789","270.1476, -30.8751",10.368,9.335,9.058,-111.7,1.047,9.423,8.732,8.674,-0.0406,Spectrum,18003541-3052303


,Aperture,"(l,b)(deg)","RA,Dec (J2000.0)",J (mag),H (mag),K (mag),Vhc (km/s),E(B-V),J0(mag),H0(mag),K0(mag),TiO(mag),Spectrum (fits),2MASS ID,0,1,0,1
0,1,"0.0058, -3.7546","270.1409, -30.8358",10.141,9.036,8.693,-67.0,1.108,9.141,8.398,8.286,-0.0316,Spectrum,18003382-3050089,0.0058,-3.7546,270.1409,-30.8358
1,2,"0.0680, -4.1923","270.6208, -30.9968",10.937,9.834,9.405,10.2,0.844,10.176,9.348,9.095,0.0971,Spectrum,18022900-3059483,0.0680,-4.1923,270.6208,-30.9968
2,3,"-0.1879, -4.0272","270.3067, -31.1382",10.493,9.369,8.901,108.3,0.867,9.711,8.870,8.583,0.4710,Spectrum,18011360-3108176,-0.1879,-4.0272,270.3067,-31.1382
3,4,"-0.0113, -3.8847","270.2630, -30.9148",10.627,9.500,9.067,43.3,0.979,9.744,8.936,8.708,0.4769,Spectrum,18010312-3054534,-0.0113,-3.8847,270.2630,-30.9148
4,7,"0.1169, -3.8291","270.2800, -30.7761",10.037,8.874,8.487,110.5,0.974,9.158,8.313,8.129,0.4879,Spectrum,18010719-3046338,0.1169,-3.8291,270.2800,-30.7761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,137,"-0.0427, -3.8945","270.2550, -30.9468",10.961,9.855,9.461,49.6,0.969,10.087,9.297,9.105,0.1758,Spectrum,18010118-3056486,-0.0427,-3.8945,270.2550,-30.9468
106,138,"0.0471, -4.2007","270.6175, -31.0191",10.697,9.631,9.134,1.4,0.829,9.949,9.154,8.830,0.4380,Spectrum,18022820-3101087,0.0471,-4.2007,270.6175,-31.0191
107,139,"-0.0860, -4.0079","270.3454, -31.0403",10.922,9.874,9.436,202.4,0.869,10.138,9.373,9.117,0.0968,Spectrum,18012290-3102249,-0.0860,-4.0079,270.3454,-31.0403
108,140,"-0.0256, -3.7789","270.1476, -30.8751",10.368,9.335,9.058,-111.7,1.047,9.423,8.732,8.674,-0.0406,Spectrum,18003541-3052303,-0.0256,-3.7789,270.1476,-30.8751


In [28]:
# Add old columns in
failed2.columns = old_cols
print(failed2.columns.values)

['Aperture' 'lb' 'RADec' 'J' 'H' 'K' 'Vhc_kmps' 'E(B-V)' 'J0' 'H0' 'K0'
 'TiO' 'Spectrum' '2MASS_ID' 'l_deg' 'b_deg' 'RA_deg' 'Dec_deg']


In [29]:
corrected2 = failed2[new_cols]
display(corrected2)

,Aperture,l_deg,b_deg,RA_deg,Dec_deg,J,H,K,Vhc_kmps,E(B-V),J0,H0,K0,TiO,2MASS_ID
0,1,0.0058,-3.7546,270.1409,-30.8358,10.141,9.036,8.693,-67.0,1.108,9.141,8.398,8.286,-0.0316,18003382-3050089
1,2,0.0680,-4.1923,270.6208,-30.9968,10.937,9.834,9.405,10.2,0.844,10.176,9.348,9.095,0.0971,18022900-3059483
2,3,-0.1879,-4.0272,270.3067,-31.1382,10.493,9.369,8.901,108.3,0.867,9.711,8.870,8.583,0.4710,18011360-3108176
3,4,-0.0113,-3.8847,270.2630,-30.9148,10.627,9.500,9.067,43.3,0.979,9.744,8.936,8.708,0.4769,18010312-3054534
4,7,0.1169,-3.8291,270.2800,-30.7761,10.037,8.874,8.487,110.5,0.974,9.158,8.313,8.129,0.4879,18010719-3046338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,137,-0.0427,-3.8945,270.2550,-30.9468,10.961,9.855,9.461,49.6,0.969,10.087,9.297,9.105,0.1758,18010118-3056486
106,138,0.0471,-4.2007,270.6175,-31.0191,10.697,9.631,9.134,1.4,0.829,9.949,9.154,8.830,0.4380,18022820-3101087
107,139,-0.0860,-4.0079,270.3454,-31.0403,10.922,9.874,9.436,202.4,0.869,10.138,9.373,9.117,0.0968,18012290-3102249
108,140,-0.0256,-3.7789,270.1476,-30.8751,10.368,9.335,9.058,-111.7,1.047,9.423,8.732,8.674,-0.0406,18003541-3052303


In [30]:
corrected2.to_csv(f"{tabdir}/BRAVA_l0.000_b-2.000.table", sep=' ', header=True, index=False, na_rep='NaN')